In [1]:
# =============================================================================
# Cell 1 - bootstrap. Rebuilds the CIC-IoT-2023 environment with a GENUINE
# support shift, per Amendment 11. Supersedes the nb32 split and ladder.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
import numpy as np, pandas as pd
ALPHA=config.ALPHA_PRIMARY
iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
assert isinstance(iot.index, pd.RangeIndex), 'prepared frame must carry a 0..n-1 RangeIndex'
rec=json.loads((config.REPORTS_DIR/'focal_class_record_ciciot2023.json').read_text())
FOCAL=rec['focal_class']; LABEL_COL=rec['label_column']
FEATS=json.loads((config.REPORTS_DIR/'ciciot2023_prepared_fingerprint.json').read_text())['features']

# Amendment 11 A11.3: fixed novel subtypes, the two smallest in the focal family
NOVEL_SUBTYPES=['Uploading_Attack','Backdoor_Malware']
print('frame:', iot.shape, '| focal:', FOCAL)
print('focal subtype counts:')
print(iot[iot.family==FOCAL]['subtype'].value_counts().to_string())
print('\nnovel (all rows to target, absent from every source partition):', NOVEL_SUBTYPES)
assert set(NOVEL_SUBTYPES) <= set(iot[iot.family==FOCAL]['subtype'].unique())


Mounted at /content/drive
frame: (1510142, 49) | focal: Web
focal subtype counts:
subtype
BrowserHijacking    5859
CommandInjection    5409
SqlInjection        5245
XSS                 3846
Backdoor_Malware    3218
Uploading_Attack    1252

novel (all rows to target, absent from every source partition): ['Uploading_Attack', 'Backdoor_Malware']


In [2]:
# =============================================================================
# Cell 2 - SOURCE / TARGET SPLIT with genuine withholding.
# Every row of a novel subtype goes to the target side. Everything else splits
# 70/30 stratified by specific label. The novel subtypes are then absent from
# train, probcal and the source calibration pool, which is what KDDTest+ gives
# NSL-KDD for free. The absence is ASSERTED, not assumed: that assertion is the
# thing whose absence produced the nb32 error.
# =============================================================================
SPLIT_SEED=20260728; TARGET_FRAC=0.30
rng=np.random.default_rng(SPLIT_SEED)
is_novel = iot['family'].eq(FOCAL) & iot['subtype'].isin(NOVEL_SUBTYPES)
print(f'novel rows forced to target: {int(is_novel.sum()):,}')

side=pd.Series(index=iot.index, dtype=object)
side.loc[is_novel]='target'
rest=iot.index[~is_novel]
for lbl,g in iot.loc[rest].groupby(LABEL_COL, sort=True):
    idx=g.index.to_numpy().copy(); rng.shuffle(idx)
    nt=int(round(TARGET_FRAC*len(idx)))
    side.loc[idx[:nt]]='target'; side.loc[idx[nt:]]='source'
iot['side']=side.values
assert iot['side'].notna().all()
print('side sizes:'); print(iot['side'].value_counts().to_string())

def strat(df, fr, seed, col):
    r=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float)
    big=nm[int(np.argmax(ff))]; a=pd.Series(index=df.index, dtype=object)
    for _, s in df.groupby(col, sort=True):
        idx=s.index.to_numpy().copy(); r.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)] += n-c.sum(); k=0
        for a2,q in zip(nm,c): a.loc[idx[k:k+q]]=a2; k+=q
    return a
src=iot[iot.side=='source']
iot['partition']=pd.Series(index=iot.index, dtype=object)
iot.loc[src.index,'partition']=strat(src, config.SPLIT_FRACTIONS, SPLIT_SEED, LABEL_COL).values
iot.loc[iot.side=='target','partition']='target_pool'
assert iot['partition'].notna().all()
print('\npartitions:'); print(iot['partition'].value_counts().to_string())

# THE ASSERTION WHOSE ABSENCE CAUSED THE nb32 ERROR
for part in ['train','val','probcal','source_cal_pool']:
    n=int(((iot.partition==part) & is_novel).sum())
    print(f'  novel rows in {part:16s}: {n}')
    assert n==0, f'novel subtypes leaked into {part}: support shift would not be real'
print('VERIFIED: novel subtypes are absent from every source partition')

print('\nfocal rows by side and subtype:')
print(iot[iot.family==FOCAL].groupby(['side','subtype']).size().unstack('side').fillna(0).astype(int).to_string())
pool=iot[iot.partition=='source_cal_pool']; need=conformal.min_calib_n(ALPHA)
print(f'\nfocal source_cal_pool rows: {int((pool.family==FOCAL).sum())} (floor {need})')
assert int((pool.family==FOCAL).sum())>=need


novel rows forced to target: 4,470
side sizes:
side
source    1053968
target     456174

partitions:
partition
train              632428
target_pool        456174
probcal            158079
source_cal_pool    158079
val                105382
  novel rows in train           : 0
  novel rows in val             : 0
  novel rows in probcal         : 0
  novel rows in source_cal_pool : 0
VERIFIED: novel subtypes are absent from every source partition

focal rows by side and subtype:
side              source  target
subtype                         
Backdoor_Malware       0    3218
BrowserHijacking    4101    1758
CommandInjection    3786    1623
SqlInjection        3671    1574
Uploading_Attack       0    1252
XSS                 2692    1154

focal source_cal_pool rows: 2135 (floor 19)


In [3]:
# =============================================================================
# Cell 3 - LADDER over genuinely unseen subtypes.
# D_eval is drawn at the SOURCE-side family prevalence so that S_lab stays near
# zero and the manipulation is support shift in isolation (Amendment 11 A11.3).
# =============================================================================
RUNGS=[0.00,0.20,0.40,0.60,0.80]
R_REAL=config.N_LADDER_REALIZATIONS['ciciot2023']
EVAL_N=60000
src_prev = (src['family'].value_counts(normalize=True))
focal_quota = int(round(EVAL_N*src_prev[FOCAL]))
print(f'source-side focal prevalence {src_prev[FOCAL]:.4%} -> {focal_quota} focal rows per D_eval')
assert focal_quota>=200

tgt=iot[iot.side=='target']
pool_novel = tgt.index[(tgt.family==FOCAL)&(tgt.subtype.isin(NOVEL_SUBTYPES))].to_numpy()
pool_seen  = tgt.index[(tgt.family==FOCAL)&(~tgt.subtype.isin(NOVEL_SUBTYPES))].to_numpy()
pool_other = tgt.index[tgt.family!=FOCAL].to_numpy()
print(f'target pools -> novel {len(pool_novel):,} | seen focal {len(pool_seen):,} | other {len(pool_other):,}')

assign=[]
for j in range(R_REAL):
    for rung in RUNGS:
        n_nov=int(round(rung*focal_quota)); n_seen=focal_quota-n_nov
        n_oth=EVAL_N-focal_quota
        assert n_nov<=len(pool_novel) and n_seen<=len(pool_seen) and n_oth<=len(pool_other), \
            f'quota unmet at realization {j} rung {rung}'
        rr=np.random.default_rng(int(hashlib.sha256(f'iot2draw|{j}|{rung}'.encode()).hexdigest(),16)%(2**32))
        pick=np.concatenate([rr.choice(pool_novel,n_nov,replace=False),
                             rr.choice(pool_seen, n_seen,replace=False),
                             rr.choice(pool_other,n_oth,replace=False)])
        assign.append(pd.DataFrame({'realization':np.full(len(pick),j,dtype=np.int16),
                                    'rung':np.full(len(pick),rung,dtype=np.float64),
                                    'row_idx':pick.astype(np.int64),
                                    'novel':'|'.join(NOVEL_SUBTYPES)}))
lad=pd.concat(assign, ignore_index=True)
print(f'\nladder rows {len(lad):,} | cells {lad.groupby(["realization","rung"]).ngroups}')

comp=lad.merge(iot[['family','subtype']], left_on='row_idx', right_index=True)
comp=comp[comp.family==FOCAL].copy()
comp['is_novel']=comp['subtype'].isin(NOVEL_SUBTYPES)
chk=comp.groupby('rung')['is_novel'].mean()
print('\nrealised NOVEL fraction of focal eval mass (this is true S_sup now):')
print(chk.round(4).to_string())
devs={r: abs(float([v for k,v in chk.items() if abs(float(k)-r)<1e-6][0])-r) for r in RUNGS}
print('per-rung deviation:', {k:round(v,4) for k,v in devs.items()})
assert max(devs.values())<0.02


source-side focal prevalence 1.3520% -> 811 focal rows per D_eval
target pools -> novel 4,470 | seen focal 6,109 | other 445,595

ladder rows 1,500,000 | cells 25

realised NOVEL fraction of focal eval mass (this is true S_sup now):
rung
0.0    0.0000
0.2    0.1998
0.4    0.3995
0.6    0.6005
0.8    0.8002
per-rung deviation: {0.0: 0.0, 0.2: 0.0002, 0.4: 0.0005, 0.6: 0.0005, 0.8: 0.0002}


In [4]:
# =============================================================================
# Cell 4 - SHIFT MEASUREMENT. S_sup is now the fraction of focal evaluation mass
# in subtypes with ZERO rows in the source, which the cell verifies directly
# rather than trusting the ladder label.
# =============================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
SUB=20000
src_subtypes=set(iot.loc[iot.side=='source','subtype'].unique())
absent=set(NOVEL_SUBTYPES)-src_subtypes
print('subtypes with zero source rows:', sorted(absent))
assert absent==set(NOVEL_SUBTYPES), 'a designated novel subtype still appears in the source'

srcpool_idx=iot.index[iot.partition=='source_cal_pool'].to_numpy()
rows=[]
for (j,rung), g in lad.groupby(['realization','rung']):
    ev=g['row_idx'].to_numpy()
    rs=np.random.default_rng(int(hashlib.sha256(f'iot2shift|{j}|{rung}'.encode()).hexdigest(),16)%(2**32))
    a=rs.choice(srcpool_idx, min(SUB,len(srcpool_idx)), replace=False)
    b=rs.choice(ev, min(SUB,len(ev)), replace=False)
    X=np.vstack([iot.loc[a,FEATS].to_numpy(float), iot.loc[b,FEATS].to_numpy(float)])
    y=np.r_[np.zeros(len(a)), np.ones(len(b))]
    aucs=[]
    for tr,te in StratifiedKFold(5, shuffle=True, random_state=0).split(X,y):
        m=HistGradientBoostingClassifier(max_iter=100, random_state=0).fit(X[tr],y[tr])
        aucs.append(roc_auc_score(y[te], m.predict_proba(X[te])[:,1]))
    s_cov=float(np.mean(aucs))
    pa=iot.loc[a,'family'].value_counts(normalize=True); pb=iot.loc[b,'family'].value_counts(normalize=True)
    fams=sorted(set(pa.index)|set(pb.index))
    s_lab=float(0.5*np.abs(np.array([pa.get(f,0) for f in fams])-np.array([pb.get(f,0) for f in fams])).sum())
    fe=iot.loc[ev]; fe=fe[fe.family==FOCAL]
    s_sup=float(fe['subtype'].isin(absent).mean()) if len(fe) else 0.0   # measured against ACTUAL absence
    rows.append({'realization':j,'rung':float(rung),'S_cov':round(s_cov,4),
                 'S_lab':round(s_lab,4),'S_sup':round(s_sup,4),'novel':'|'.join(sorted(absent))})
    print(f'  r{j} rung {rung:.2f}: S_cov={s_cov:.4f} S_lab={s_lab:.4f} S_sup={s_sup:.4f}')
shift=pd.DataFrame(rows)
print('\nby rung (mean over realizations):')
print(shift.groupby('rung')[['S_cov','S_lab','S_sup']].mean().round(4).to_string())
print(f'\nS_cov {shift.S_cov.min():.4f}-{shift.S_cov.max():.4f} | '
      f'S_lab {shift.S_lab.min():.4f}-{shift.S_lab.max():.4f} | '
      f'S_sup {shift.S_sup.min():.4f}-{shift.S_sup.max():.4f}')


subtypes with zero source rows: ['Backdoor_Malware', 'Uploading_Attack']
  r0 rung 0.00: S_cov=0.4969 S_lab=0.0113 S_sup=0.0000
  r0 rung 0.20: S_cov=0.4972 S_lab=0.0045 S_sup=0.1998
  r0 rung 0.40: S_cov=0.5006 S_lab=0.0116 S_sup=0.3995
  r0 rung 0.60: S_cov=0.5037 S_lab=0.0069 S_sup=0.6005
  r0 rung 0.80: S_cov=0.5042 S_lab=0.0077 S_sup=0.8002
  r1 rung 0.00: S_cov=0.5003 S_lab=0.0138 S_sup=0.0000
  r1 rung 0.20: S_cov=0.5010 S_lab=0.0038 S_sup=0.1998
  r1 rung 0.40: S_cov=0.4953 S_lab=0.0061 S_sup=0.3995
  r1 rung 0.60: S_cov=0.5034 S_lab=0.0077 S_sup=0.6005
  r1 rung 0.80: S_cov=0.5062 S_lab=0.0040 S_sup=0.8002
  r2 rung 0.00: S_cov=0.5007 S_lab=0.0082 S_sup=0.0000
  r2 rung 0.20: S_cov=0.5034 S_lab=0.0073 S_sup=0.1998
  r2 rung 0.40: S_cov=0.4956 S_lab=0.0038 S_sup=0.3995
  r2 rung 0.60: S_cov=0.5017 S_lab=0.0113 S_sup=0.6005
  r2 rung 0.80: S_cov=0.5027 S_lab=0.0079 S_sup=0.8002
  r3 rung 0.00: S_cov=0.4961 S_lab=0.0055 S_sup=0.0000
  r3 rung 0.20: S_cov=0.4954 S_lab=0.0070 S_sup

In [5]:
# =============================================================================
# Cell 5 - rename the superseded artefacts, persist the corrected ones, commit.
# Amendment 11 A11.2: the nb32/nb34 run is kept as a no-shift control.
# =============================================================================
RD=config.REPORTS_DIR; PD=config.PROC_DIR
for old,new in [(RD/'coverage_primary_ciciot2023.csv', RD/'coverage_noshift_control_ciciot2023.csv'),
                (RD/'ladder_shift_measures_ciciot2023.csv', RD/'noshift_control_measures_ciciot2023.csv'),
                (PD/'ciciot2023_ladder_assignments.parquet', PD/'ciciot2023_noshift_ladder.parquet'),
                (PD/'ciciot2023_split.parquet', PD/'ciciot2023_noshift_split.parquet')]:
    if old.exists() and not new.exists(): old.rename(new); print('renamed', old.name, '->', new.name)

iot[['side','partition']].to_parquet(PD/'ciciot2023_split.parquet')
lad.to_parquet(PD/'ciciot2023_ladder_assignments.parquet', index=False)
shift.to_csv(RD/'ladder_shift_measures_ciciot2023.csv', index=False)
(RD/'ciciot2023_split_record.json').write_text(json.dumps({
  'amendment':'preregistration_amendment_11.md','split_seed':SPLIT_SEED,
  'target_frac':TARGET_FRAC,'novel_subtypes':NOVEL_SUBTYPES,
  'novel_rows_all_to_target':int(is_novel.sum()),
  'novel_absent_from_source_verified':True,
  'focal_class':FOCAL,'focal_source_cal_pool':int((pool.family==FOCAL).sum()),
  'rungs':RUNGS,'realizations':R_REAL,'eval_n':EVAL_N,
  'eval_prior':'source-side family prevalence, so S_lab stays near zero and the manipulation '
               'is support shift in isolation',
  'focal_quota_per_eval':focal_quota,
  'supersedes':'the nb32 split and ladder, retained as the no-shift control per A11.2'
}, indent=2, default=str))
print('\nsaved corrected split, ladder and shift measures')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb32b: CIC-IoT-2023 corrected split with genuinely withheld subtypes (Amendment 11); prior run retained as no-shift control')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)
print('\nNEXT: re-run nb33 (models) then nb34 (coverage). Delete data/ciciot_probs/*.npz first:')
print('  the source side has changed, so the cached probabilities are stale.')


renamed coverage_primary_ciciot2023.csv -> coverage_noshift_control_ciciot2023.csv
renamed ladder_shift_measures_ciciot2023.csv -> noshift_control_measures_ciciot2023.csv
renamed ciciot2023_ladder_assignments.parquet -> ciciot2023_noshift_ladder.parquet
renamed ciciot2023_split.parquet -> ciciot2023_noshift_split.parquet

saved corrected split, ladder and shift measures
[main 1bd3f41] nb32b: CIC-IoT-2023 corrected split with genuinely withheld subtypes (Amendment 11); prior run retained as no-shift control
 7 files changed, 128 insertions(+), 53 deletions(-)
 create mode 100644 notebooks/32b_ciciot2023_corrected_split_ladder.ipynb
 create mode 100644 preregistration_amendment_11.md
 rewrite reports/ciciot2023_split_record.json (76%)
 rename reports/{coverage_primary_ciciot2023.csv => coverage_noshift_control_ciciot2023.csv} (100%)
 rewrite reports/ladder_shift_measures_ciciot2023.csv (100%)
 copy reports/{ladder_shift_measures_ciciot2023.csv => noshift_control_measures_ciciot2023.csv} 